# Epoch-Based Extreme Rainfall Frequency Comparison

**Purpose:**  
Quantify changes in extreme rainfall event frequency across historical time windows (“epochs”) using observed PRISM data.

This notebook implements the same epoch-based comparison logic used in the thesis analysis, generalized for replication and reuse across regions.

**Key Concept:**  
Climate non-stationarity is expressed here as **changes in event frequency over time**, not shifts in intensity or fitted return-period models.

**Inputs:**  
- Annual extreme-event counts generated in `02_prism_extremes_analysis.ipynb`

**Outputs:**  
- Epoch-level event totals  
- Absolute and percent frequency change metrics  
- Tables suitable for mapping, reporting, and comparison


## Interpretation Notes

- Epoch comparisons reflect **observed changes in event frequency**, not probabilistic return periods.
- Percent change values are sensitive to low baseline counts and should be interpreted alongside absolute totals.
- This step preserves the empirical framing of the original thesis while improving transparency and reuse.

Results from this notebook feed into:
- future frequency comparison (Notebook 04)
- exposure and vulnerability overlays (Notebook 05)
- flood probability context (Notebook 06)


In [1]:
# ----------------------------------
# Repo-aware path setup (REQUIRED)
# ----------------------------------
import sys
from pathlib import Path
import pandas as pd

# Resolve repository root (notebooks/ is one level down)
REPO_ROOT = Path("..").resolve()

# Ensure src/ is importable (future-proofing)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Core repo directories
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DATA_SAMPLE_DIR = DATA_DIR / "sample"

OUTPUTS_DIR = REPO_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
MAPS_DIR = OUTPUTS_DIR / "maps"

DOCS_DIR = REPO_ROOT / "docs"
SAMPLE_DIR = REPO_ROOT / "sample_data"

# Ensure output directories exist
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Tables dir:", TABLES_DIR)

Repo root: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework
Tables dir: C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework\outputs\tables


In [2]:
# -----------------------------
# USER CONFIGURATION
# -----------------------------

# Input from Notebook 02
INPUT_PATH = TABLES_DIR / "prism_annual_extreme_counts.csv"

# Output table
OUTPUT_PATH = TABLES_DIR / "epoch_comparison_summary.csv"

# Epoch length (years)
EPOCH_YEARS = 20

In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing annual extremes table at {INPUT_PATH}. "
        "Run 02_prism_extremes_analysis.ipynb first."
    )

df = pd.read_csv(INPUT_PATH)
df.head()

,year,extreme_event_count
0,1981,0
1,1982,0
2,1983,0
3,1984,0
4,1985,0


In [4]:
# Ensure year is integer
df["year"] = df["year"].astype(int)

df.head()

,year,extreme_event_count
0,1981,0
1,1982,0
2,1983,0
3,1984,0
4,1985,0


In [5]:
# -----------------------------
# EPOCH BOUNDS
# -----------------------------

START_YEAR = df["year"].min()
END_YEAR = df["year"].max()

START_YEAR, END_YEAR

(np.int64(1981), np.int64(2021))

In [6]:
# Assign each year to an epoch start year
df["epoch_start"] = (
    ((df["year"] - START_YEAR) // EPOCH_YEARS) * EPOCH_YEARS
) + START_YEAR

df[["year", "epoch_start", "extreme_event_count"]].head(10)

,year,epoch_start,extreme_event_count
0,1981,1981,0
1,1982,1981,0
2,1983,1981,0
3,1984,1981,0
4,1985,1981,0
5,1986,1981,0
6,1987,1981,0
7,1988,1981,0
8,1989,1981,0
9,1990,1981,2


In [7]:
# Aggregate extreme-event counts by epoch
epoch_summary = (
    df.groupby("epoch_start", as_index=False)
      .agg(epoch_event_total=("extreme_event_count", "sum"))
)

epoch_summary

,epoch_start,epoch_event_total
0,1981,4
1,2001,8
2,2021,0


In [8]:
# Calculate frequency change metrics
epoch_summary = epoch_summary.sort_values("epoch_start")

epoch_summary["previous_epoch_total"] = epoch_summary["epoch_event_total"].shift(1)

epoch_summary["absolute_change"] = (
    epoch_summary["epoch_event_total"]
    - epoch_summary["previous_epoch_total"]
)

epoch_summary["percent_change"] = (
    epoch_summary["absolute_change"]
    / epoch_summary["previous_epoch_total"].replace(0, pd.NA)
) * 100

epoch_summary

,epoch_start,epoch_event_total,previous_epoch_total,absolute_change,percent_change
0,1981,4,NaN,NaN,NaN
1,2001,8,4.0,4.0,100.0
2,2021,0,8.0,-8.0,-100.0


In [9]:
epoch_summary.to_csv(OUTPUT_PATH, index=False)

print(f"Epoch comparison table saved to {OUTPUT_PATH}")

Epoch comparison table saved to C:\Users\admin\Documents\GitHub\nonstationary-flood-risk-framework\outputs\tables\epoch_comparison_summary.csv


## Next Steps

This notebook compares annual extreme rainfall event counts across historical time windows (“epochs”) to identify observed changes in event frequency without assuming stationarity.

The epoch-based frequency change metrics generated here are used in:

- `04_cordex_future_frequency.ipynb`  
  → to compare observed frequency shifts with projected future changes from climate models

Users may proceed directly to the next notebook after confirming this step completes successfully.